# Notebook to create .npy references used for evaluation of SAR_DDC


In [ ]:
from pathlib import Path
import sys
from omegaconf import OmegaConf
import hydra
import matplotlib.pyplot as plt

sys.path.append(str(Path().resolve().parent))
import numpy as np
import torch
from src.utils.constants import amp_min, amp_max
from src.utils.sar_utils import symmetrize
from src.utils.processing_utils import process_large_patch

STORAGE_PATH = Path("../data/visualization/for_evaluations/")
CKP_PATH = Path("../logs/train/sar_ddc/")

MERLIN_CKP_PATH = CKP_PATH / "merlin/runs/2025-09-16_13-49-03/checkpoints/last.ckpt"
ADAM_NOC_CKP_PATH = CKP_PATH / "hyperprior/runs/2025-09-17_08-55-41/checkpoints/last.ckpt"

HAMBURG_TILE_PATH = STORAGE_PATH.parent / "raw_Hamburg_patch_[11000:12024;8500:9524].npy"

EPS = 1e-2
STRIDE = 128
BLEND_METHOD = "count"


def print_stats(name: str, arr: np.ndarray) -> None:
    print(
        f"{name: <20}: (shape={arr.shape}) statistics: min={arr.min():<10.3f}, max={arr.max():<10.3f}, mean={arr.mean():<10.3f}, std={arr.std():<10.3f}. Is NaN={np.isnan(arr).any()}."
    )


def clip(arr: np.ndarray, factor: int = 3) -> np.ndarray:
    m = arr.mean()
    s = arr.std()
    return np.clip(arr, m - factor * s, m + factor * s)


def show_image(
    im: np.ndarray,
    title: str,
    clip_factor: int = 3,
    dpi: int = 300,
    path_to_png: Path | None = None,
) -> None:
    im_clipped = clip(im, factor=clip_factor)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=dpi)
    ax.imshow(im_clipped, cmap="gray")
    ax.set_title(title)
    ax.axis("off")
    fig.show()

    if path_to_png is not None:
        plt.imsave(path_to_png, im_clipped, cmap="gray", dpi=dpi)
        print(f"Saved image to {path_to_png}.")

In [ ]:
# --- Load and process Hamburg patch ---
patch = np.load(HAMBURG_TILE_PATH)  # [H, W, 2]
print(f"Loaded RAW PATCH from {HAMBURG_TILE_PATH}.")
print_stats("Raw patch", patch)

patch = symmetrize(patch)
print_stats("Symmetrized patch", patch)

patch = torch.from_numpy(patch)
patch = torch.square(patch)
patch = torch.log(patch + EPS)
patch = (patch - 2 * amp_min) / (2 * amp_max - 2 * amp_min)
real = patch[..., 0].unsqueeze(0).unsqueeze(0)
imag = patch[..., 1].unsqueeze(0).unsqueeze(0)
print_stats("Log-normalized patch", patch.numpy())

## MERLIN (own implementation)

In [ ]:
# Find and load the config of MERLIN
merlin_config_path = MERLIN_CKP_PATH.parent.parent / ".hydra" / "config.yaml"
if not merlin_config_path.exists():
    raise FileNotFoundError(f"Training config not found at {merlin_config_path}.")
print(f"Loading original MERLIN training config from {merlin_config_path}")
merlin_cfg = OmegaConf.load(merlin_config_path)

# Load MERLIN and generate the references for the Hamburg tile
print(f"Instantiating model <{merlin_cfg.model._target_}>")
merlin_model = hydra.utils.instantiate(merlin_cfg.model)
merlin_checkpoint = torch.load(str(MERLIN_CKP_PATH), map_location="cpu")
msg = merlin_model.load_state_dict(merlin_checkpoint["state_dict"], strict=True)
print(f"Loaded MERLIN checkpoint state_dict with message: {msg}")
merlin_model.eval()

criterion_real, recon_real = process_large_patch(
    model=merlin_model,
    input=real,
    target=None,
    stride=STRIDE,
    blend_method=BLEND_METHOD,
)
print_stats("MERLIN reconstruction real", recon_real.numpy())
criterion_imag, recon_imag = process_large_patch(
    model=merlin_model,
    input=imag,
    target=None,
    stride=STRIDE,
    blend_method=BLEND_METHOD,
)
print_stats("MERLIN reconstruction imag", recon_imag.numpy())

recon_real = torch.exp(recon_real.squeeze() * (amp_max - amp_min) + amp_min)
recon_imag = torch.exp(recon_imag.squeeze() * (amp_max - amp_min) + amp_min)
merlin_recon = 0.5 * (recon_real + recon_imag)
merlin_recon_logI = torch.log(merlin_recon + EPS).numpy()
print_stats("MERLIN reconstruction Log-I", merlin_recon_logI)

# Save the references
np.save(
    STORAGE_PATH / "denoised_by_MERLIN_[11000:12024;8500:9524]_logI.npy",
    merlin_recon_logI,
)

In [ ]:
# Load the saved image, print stats and visualize
merlin_path = STORAGE_PATH / "denoised_by_MERLIN_[11000:12024;8500:9524]_logI.npy"
merlin_ref = np.load(merlin_path)
print_stats("Loaded MERLIN reference", merlin_ref)

show_image(
    merlin_ref,
    "Loaded MERLIN Reference",
    dpi=150,
    path_to_png=merlin_path.with_suffix(".png"),
)

## ADAM no compression

In [ ]:
def evaluate_and_visualize_ADAM_ckpt(
    ckpt_path: Path, lmbda: int, real: torch.Tensor, imag: torch.Tensor
) -> None:
    if not ckpt_path.exists():
        raise FileNotFoundError(f"Training config not found at {ckpt_path}.")
    print(f"Loading original ADAM training config from {ckpt_path}")
    adam_cfg = OmegaConf.load(ckpt_path)

    # Load ADAM and generate the references for the Hamburg tile
    print(f"Instantiating model <{adam_cfg.model._target_}>")
    adam_model = hydra.utils.instantiate(adam_cfg.model)
    adam_checkpoint = torch.load(str(ADAM_NOC_CKP_PATH), map_location="cpu")
    msg = adam_model.load_state_dict(adam_checkpoint["state_dict"], strict=True)
    print(f"Loaded ADAM checkpoint state_dict with message: {msg}")
    adam_model.eval()

    criterion_real, recon_real = process_large_patch(
        model=adam_model,
        input=real,
        target=None,
        stride=STRIDE,
        blend_method=BLEND_METHOD,
    )
    print_stats("ADAM reconstruction real", recon_real.numpy())
    criterion_imag, recon_imag = process_large_patch(
        model=adam_model,
        input=imag,
        target=None,
        stride=STRIDE,
        blend_method=BLEND_METHOD,
    )
    print_stats("ADAM reconstruction imag", recon_imag.numpy())

    recon_real = torch.exp(recon_real.squeeze() * (amp_max - amp_min) + amp_min)
    recon_imag = torch.exp(recon_imag.squeeze() * (amp_max - amp_min) + amp_min)
    adam_recon = 0.5 * (recon_real + recon_imag)
    adam_recon_logI = torch.log(adam_recon + EPS).numpy()
    print_stats("ADAM reconstruction Log-I", adam_recon_logI)

    show_image(
        adam_recon_logI,
        f"Loaded ADAM {lmbda}",
        dpi=150,
        path_to_png=STORAGE_PATH
        / f"denoised_by_ADAM-lambda{lmbda}_[11000:12024;8500:9524]_logI.png",
    )

In [ ]:
adam_config_path = ADAM_NOC_CKP_PATH.parent.parent / ".hydra" / "config.yaml"
evaluate_and_visualize_ADAM_ckpt(ckpt_path=adam_config_path, lmbda=0, real=real, imag=imag)

## Other ADAM

In [ ]:
# Define a map from pathes to lambda values
basic_path = Path("logs/train/sar_ddc/hyperprior/")
other_adam_configs = {
    basic_path / "runs/2025-09-18_13-05-43/.hydra/config.yaml": 1,
    basic_path / "runs/2025-09-17_20-38-04/.hydra/config.yaml": 10,
    basic_path / "runs/2025-09-17_15-48-07/.hydra/config.yaml": 50,
    basic_path / "runs/2025-09-17_14-32-17/.hydra/config.yaml": 100,
    basic_path / "multiruns/2025-09-18_18-06-51/2/.hydra/config.yaml": 200,
}


for path, lmbda in other_adam_configs.items():
    evaluate_and_visualize_ADAM_ckpt(ckpt_path=path, lmbda=lmbda, real=real, imag=imag)

## NOISY

In [ ]:
# Load and visualize the noisy patch
noisy_path = STORAGE_PATH.parent / "noisy_Hamburg_patch_[11000:12024;8500:9524].npy"
noisy = np.load(noisy_path)
print_stats("Loaded Noisy Hamburg Patch", noisy)

noisy_ref = np.square(noisy[..., 0]) + np.square(noisy[..., 1])
noisy_ref = np.log(noisy_ref + EPS)
print_stats("Log-Intensity Noisy Hamburg Patch", noisy_ref)

show_image(
    noisy_ref,
    "Log-Intensity Noisy Hamburg Patch",
    dpi=150,
    path_to_png=noisy_path.with_suffix(".png"),
)

## MERLIN (from deepdespeckling checkpoint)

In [ ]:
# Load MERLIN_deepdespeckling
merlin_deepdespeckling_path = (
    STORAGE_PATH.parent / "denoised_by_MERLIN_Hamburg_patch_[11000:12024;8500:9524].npy"
)
merlin_deepdespeckling_ref = np.load(
    merlin_deepdespeckling_path,
    allow_pickle=True,
).item()
amp_linear = merlin_deepdespeckling_ref["denoised"]["full"]
merlin_deepdespeckling_ref = np.log(np.square(amp_linear) + EPS)
print_stats("Loaded MERLIN Reference", merlin_deepdespeckling_ref)

np.save(
    STORAGE_PATH / "denoised_by_MERLIN-DDS_[11000:12024;8500:9524]_logI.npy",
    merlin_deepdespeckling_ref,
)

show_image(
    merlin_deepdespeckling_ref,
    "Loaded MERLIN Reference",
    dpi=150,
    path_to_png=merlin_deepdespeckling_path.with_suffix(".png"),
)

In [ ]:
# Compare all refs together: subplot 2x2
fig, axs = plt.subplots(2, 2, figsize=(10, 10), dpi=300)
axs[0, 0].imshow(clip(noisy_ref, factor=3), cmap="gray")
axs[0, 0].set_title("Noisy")
axs[0, 0].axis("off")

axs[0, 1].imshow(clip(adam_noc_ref, factor=3), cmap="gray")
axs[0, 1].set_title("ADAM-NOC")
axs[0, 1].axis("off")

axs[1, 0].imshow(clip(merlin_ref, factor=3), cmap="gray")
axs[1, 0].set_title("MERLIN")
axs[1, 0].axis("off")

axs[1, 1].imshow(clip(merlin_deepdespeckling_ref, factor=3), cmap="gray")
axs[1, 1].set_title("MERLIN Deep Despeckling")
axs[1, 1].axis("off")

fig.suptitle("Comparison of Denoising References", fontsize=16)
fig.tight_layout()
fig.show()